<img src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/HydroSARbanner.jpg" width="100%" />

<br>
<font size="6"> <b>FIER Daily Flood Forecasting Code</b><img style="padding: 7px" src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/UAFLogo_A_647.png" width="170" align="right"/></font>

<br>
<font size="4"> <b> Franz J Meyer, University of Alaska Fairbanks</b> <br>
</font>

This notebook uses polynomial functions and neural network models to generate daily flood inundation predictions using time series of Sentinel-1 RTC data and GEOGLoWs river runoff forecasts. 
    
The workflow utilizes information available in the fierpy <a href="https://github.com/SERVIR/fierpy">fierpy</a> GitHub repository.
<hr>


# Load Python Libraries

In [ ]:
# load needed modules
import _pickle as cPickle
from pathlib import Path
import time
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rasterio
import re
from datetime import datetime, date
import pandas as pd
import os, datetime
import rioxarray as rxr
import csv
from datetime import datetime
import copy

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.layers import Normalization
print(tf.__version__)

In [ ]:
import geoglows
print(geoglows.__version__)

***
## Select working folder

In [ ]:
from ipyfilechooser import FileChooser
fc = FileChooser(Path.cwd())
display(fc)

***
## Load saved variables

In [ ]:
# Load the variables
start_reload = time.time()


if 'reof_ds' in globals():
    del reof_ds, modes_poly_smoothed, degs_smoothed, polys_smoothed, models_nn_smoothed, modes_nn_smoothed, q_smoothed, q_tot_smoothed, type_file, da_tot, ind_flood, floodpercent, time_flood
    print('Deleting old data...')
else:
    print('Loading data...')


forecasting_files = "ForecastingFiles_ReachID-440985761_Created-20250603_NSE=p001_forwardSmoothed_allModes-REOF_10Modes-Fit_noNovOrDec_despiked.pkl"

with open(Path(fc.selected,"Forecasting-Files",forecasting_files), "rb") as f: # for bangladesh-West
    reof_ds = cPickle.load(f) # works
    print('reof_ds complete')
    modes_poly_smoothed = cPickle.load(f) # works
    print('modes_poly_smoothed complete')
    degs_smoothed = cPickle.load(f) # works
    print('degs_smoothed complete')
    polys_smoothed = cPickle.load(f) # works
    print('polys_smoothed complete')
    models_nn_smoothed = cPickle.load(f) # works
    print('models_nn_smoothed complete')
    modes_nn_smoothed = cPickle.load(f) # works
    print('modes_nn_smoothed complete')
    q_smoothed = cPickle.load(f) # works
    print('q_smoothed complete')
    q_tot_smoothed = cPickle.load(f) # works
    print('q_tot_smoothed complete')
    type_file = cPickle.load(f) # works
    print('type_file complete')
    da_tot = cPickle.load(f) # works
    print('da_tot complete')
    ind_flood = cPickle.load(f) # works
    print('ind_flood complete')
    floodpercent = cPickle.load(f) # works
    print('floodpercent complete')
    time_flood = cPickle.load(f) # works
    print('time_flood complete')
    lat = cPickle.load(f)
    lon = cPickle.load(f)
    print('lat & lon complete')
    ReachID = cPickle.load(f)
    print('ReachID complete')
    model_sel = cPickle.load(f)
    print('Model selection complete')
    CRS = cPickle.load(f)
    print('CRS complete')
    transform = cPickle.load(f)
    print('transform complete')
    smoothing_frame = cPickle.load(f)
    print('smoothing_frame loaded')
    scores_nn = cPickle.load(f)
    print('scores_nn loaded')
    scores_poly = cPickle.load(f)
    print('scores_poly loaded')
    NSEs_poly_smoothed = cPickle.load(f)
    print('NSEs_poly_smoothed loaded')
    NSEs_nn_smoothed = cPickle.load(f)
    print('NSEs_nn_smoothed loaded')
    myIdx = cPickle.load(f)
    print('myIdx loaded')


end_reload = time.time()

print('It took ',end_reload-start_reload,' seconds to load the variables.')

***
## Select Model Parameters (UNDER CONSTRUCTION)

* myLoc
    * Generalized location to save the figures and output data.
    * The location is based on Path(fc.selected,'Figures',myLoc).
* debuff

In [ ]:
# generalized location in which to save files
myLoc = f'forwardSmoothed-trainingOnly_noNovOrDec_Despiked_debuffed_discharge_{datetime.today().strftime('%Y-%m-%d')}'
# myLoc = Path('Geoglows-Comparison','geoglows-v1p7p0','NSE=0p2')
myLoc = Path('Geoglows-Comparison','geoglows-v1p7p0','NSE=0p001')
myLoc = Path('June2025-workshop')

debuff = False
if debuff:
    debuff_shapefile = FileChooser(Path.cwd())
    display(debuff_shapefile)

# set y axis limits for plots flood percentage (observed and forecasted) & discharge vs time; this allows for intercomparison between different runs. 
# This can be set to 'None' (which is the default) for y limits to automatically scale.
fp_axis_lims = [0, 35]

In [ ]:
print(f'Your save location: {Path(fc.selected, 'Figures', myLoc)}')
if debuff:
    print(f'\nYour debuff_shapefile location: {debuff_shapefile.selected}')

## Select smoothing to use

In [ ]:
myIdx_original = copy.deepcopy(myIdx)
print(myIdx_original)

In [ ]:
print(f'max allowable value for myIdx = {len(smoothing_frame)-1}')

In [ ]:
import sys

myIdx = 5 # this will be changed from 0 up to the maximum number of smoothings

if myIdx > len(smoothing_frame)-1:
    raise Exception(f'\nWARNING!!!\nWARNING!!!\nWARNING!!!\nmyIdx is greater than the highest smoothing frame index!\nmyIdx = {myIdx}, max frame index = {len(smoothing_frame)-1}')

if len(smoothing_frame) >= 2:
    # q_tot = q_tot_smoothed[myIdx].copy(deep=True) # I AM HERE; try this and see how it affects selected hindcast (the 'forecast' that uses historical data to test the quality of the fitting)
    q_tot = q_tot_smoothed[0].copy(deep=True) # 
else:
    q_tot = q_tot_smoothed.copy(deep=True)
    
# polynomial
modes_poly = copy.deepcopy(modes_poly_smoothed[myIdx])
degs = copy.deepcopy(degs_smoothed[myIdx])
polys = copy.deepcopy(polys_smoothed[myIdx])

# neural network
modes_nn = copy.deepcopy(modes_nn_smoothed[myIdx])
models_nn = copy.deepcopy(models_nn_smoothed[myIdx])

## Define important functions

In [ ]:
### --- Z-Score calculation --- ###
def z_score(flood_forecast, flood_hindcast):
    # Calculate the indices for scoring: a = true positive, b = false positive, c = false negative, d = true negative
    a = np.sum(np.logical_and(flood_forecast == 1, flood_hindcast == 1))
    b = np.sum(np.logical_and(flood_forecast == 1, flood_hindcast == 0))
    c = np.sum(np.logical_and(flood_forecast == 0, flood_hindcast == 1))
    d = np.sum(np.logical_and(flood_forecast == 0, flood_hindcast == 0))

    # Calculate the skills
    overall_accuracy = ((a + d) / (a + b + c + d)) * 100
    CSI = (a / (a + b + c)) * 100
    precision = (a / (a + b)) * 100
    recall = (a / (a + c)) * 100
    
    # Replace NaNs by 0s
    if np.isnan(overall_accuracy):
        overall_accuracy = 0
    if np.isnan(CSI):
        CSI = 0
    if np.isnan(precision):
        precision = 0
    if np.isnan(recall):
        recall = 0

    return overall_accuracy, CSI, precision, recall


### --- Forecast calculation --- ###

# define function to create the forecast for a single mode for the single mode or the multi-mode forecast
# The only difference is that in multi-mode, the mean (reof_ds.center.values) is added AFTER all of the modes have been caluclated and added up. 
def myForecaster(forecastType, allModes, myMode, q_forecast, reof_ds, models, type_file, da_tot, singleOrMulti): 
    # Create the forecast based on a single spatial mode
    if 's' in singleOrMulti:
        print(f'Single-mode forecast initiated.\nRunning for mode {myMode}...')
    elif 'm' in singleOrMulti: 
        print(f'Multi-mode forecast initiated.\nRunning for mode {myMode}...')

    # Get dimensions of datasets
    n_modes, y, x = reof_ds.spatial_modes.shape
    t = q_forecast.shape[0]

    # Create the variable filled with zeros
    zeros_variable = np.zeros((t, y, x))

    ### --- Initialize some important arrays --- ###
    
    # Create the new DataArray with zeros_variable
    da = xr.DataArray(
        zeros_variable,
        coords={'time': q_forecast.time, 'y': da_tot.y, 'x': da_tot.x},
        dims=['time', 'y', 'x']
    )

    # Make predictions using the best model(s)
    if 'p' in forecastType:
        forecast_rtcp = np.array([model(q_forecast.values) for model in models]).squeeze()
    elif 'n' in forecastType:
        forecast_rtcp = np.array([model.predict(q_forecast.values) for model in models]).squeeze() # original
    else:
        print('Input forecastType is neither neural network nor polynomial!')

    # Check the size of the forecasted rtcps. If it has only one dimension, add a new dimension
    if len(forecast_rtcp.shape) == 1:
        forecast_rtcp = forecast_rtcp[:, np.newaxis] # Add a new dimension
        forecast_rtcp = forecast_rtcp.T # Transpose to match the dimensions of the spatial modes

    ### --- Reconstruct the dataset by multiplying the spatial mode by the forecast for each significant mode --- ###
    
    idxM = allModes.index(myMode)
    forecast_nocenter = np.dot(reof_ds.spatial_modes.sel(mode=allModes[idxM]).values.reshape(-1,1), forecast_rtcp[idxM].reshape(1,len(forecast_rtcp[idxM]))).reshape(y,x,t).transpose(2,0,1)
    for i in range(0,forecast_nocenter.shape[0]):
        forecast_nocenter[i,:,:] = np.flipud(forecast_nocenter[i,:,:])
    

    # Add to the reconstructed dataset
    if 's' in singleOrMulti:
        da.values += forecast_nocenter + reof_ds.center.values
    elif 'm' in singleOrMulti: 
        da.values += forecast_nocenter

    # If water mask, REOF gives FLOAT that we have to round to 0 or 1
    if type_file == 'Water_Masks':
        da.values = np.round(da.values)

    if 's' in singleOrMulti: 
        # change values to 0 (no water) or 1 (water) as appropriate
        # anything greater than 0.5 and less than 255 is water. 
        idxs = (da >= 0.5) * (da < 255)
        da.values[idxs] = 1.0
        # anything greater than or equal to 255 is a NaN (fully converted later)
        idxs = (da >= 255)
        da.values[idxs] = 255
        # anything less than 0.5 is made to zero (not water). 
        idxs = (da < 0.5)
        da.values[idxs] = 0.0

    # Convert 255 to NaN
    da.values = np.where(da.values==255, np.nan, da.values)
    
    return da

### --- Visualization code --- ###


# define function to get flood percentages from each flood forecast. 
def floodPercentCalc(forecast): 
    # set up empty array into which to put the percentages
    floodPercent=np.array([])
    # set up the total number of forecasts and a counter to keep track of progress. 
    total = forecast.shape[0]
    county = 0
    print("Starting calcualtion")
    for cast in forecast:
        # get the unique elements and their counts
        unique_elements, counts = np.unique(cast, return_counts=True)
        myDict = dict(zip(unique_elements, counts))
        myArea = myDict[0] + myDict[1]
        flooded = myDict[1]
        floodPercent = np.append(floodPercent, flooded/myArea*100.0)
        county += 1
        print(f"{county} of {total} complete")
    return floodPercent


# define function to plot flood percentage and discharge through time
def plot_flood_and_discharge(forecasts, flood_percents, q_tot, floodpercent_real = None, time_real=None, saveIt=False, fp_lims = None):
    
    # set up figure and twin the axis
    fig_perc_dis, ax1 = plt.subplots(figsize=(9,6))
    ax2 = ax1.twinx()

    
    # plot the neural network flood percentages
    ln1 = ax1.plot(forecasts.time, flood_percents, color='blue', marker='o', label='Flood Percentage from Forecast')
    ax1.set_ylabel('Flood Percentage [%]')
    if fp_lims:
        ax1.set_ylim(fp_lims[0], fp_lims[1])
    ax1.grid()
    
    # plot the discharge values through same time. 
    idx1, idx2 = np.where(forecasts[0].time == q_tot.time)[0][0], np.where(forecasts[-1].time == q_tot.time)[0][0]
    ln2 = ax2.plot(q_tot[idx1:idx2+1].time, q_tot[idx1:idx2+1], color='red', marker='^', label='Discharge from GEOGLOWS')
    ax2.set_ylabel('Discharge [$m^3/s$]')

    lns = ln1 + ln2

    # plot real data if available
    if floodpercent_real is not None:
        # define function to get index of nearest observed flood event. 
        def nearest_ind(items, pivot):
            time_diff = np.abs([date - pivot for date in items])
            return time_diff.argmin(0)
        # get nearest observed flood event
        idxA = nearest_ind(time_real, q_tot[0].time.values)
        idxB = nearest_ind(time_real, q_tot[-1].time.values)
        # plot observed flood discharges
        ln3 = ax1.plot(time_real[idxA:idxB+1], floodpercent_real[idxA:idxB+1], color='green', marker='x', markersize=10, linestyle='None', label='Flood Percentage from SAR')
        lns = lns + ln3

    # create legend entries
    labs = [l.get_label() for l in lns]
    ax1.legend(lns, labs)

    plt.setp(ax1.get_xticklabels(), rotation=45)
    
    # save figures
    if saveIt:
        Path(fc.selected,'Figures').mkdir(parents=True, exist_ok=True)
        plt.savefig(Path(fc.selected,'Figures','floodANDdischargeVStime.tif'), format='tiff', dpi=300)

    
    plt.show()

    return fig_perc_dis



# define function that will run the comparison
def forecast2observed_comparison(da_forecast, da_observed, myCmap='gray', myType=[], myDate=[]):

    ### --- Get statistics --- ###
    # Calculate statistics
    overall_accuracy, CSI, precision, recall = z_score(da_forecast.values, da_observed.values)
    difference_sum = np.nansum(abs(da_forecast.values - da_observed.values))
    difference_percent = difference_sum / np.count_nonzero(~np.isnan(da_forecast.values)) * 100.0
    # Print statistics
    print(f"Overall accuracy: {overall_accuracy:.2f}%")
    print(f"Critical Success Index: {CSI:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall: {recall:.2f}%")
    print(f"Difference Summation: {difference_sum:.0f}")
    print(f"Difference Percent: {difference_percent:.2f}%")
    info = f"\n\nOverall accuracy: {overall_accuracy:.2f}% \nCritical Success Index: {CSI:.2f}% \nPrecision: {precision:.2f}% \nRecall: {recall:.2f}% \nDifference Summation: {difference_sum:.0f} \nDifference Percent: {difference_percent:.2f}%"
    

    ### --- Plot the forecasts --- ###

    
    # Plot the difference between the forecast and real image
    fig_results, ax_results = plt.subplots(1,3, figsize=(20,5))
    # im = ax_results[0].imshow(np.round(da_forecast.values - da_observed.values), vmin=-1, vmax=1, cmap = 'seismic_r', interpolation=None)
    im = ax_results[0].imshow(da_forecast.values - da_observed.values, vmin=-1, vmax=1, cmap = 'seismic_r', interpolation=None)
    plt.colorbar(im, ax=ax_results[0])
    ax_results[0].set_title('Difference Map\nBlue=extra, Red=missing')
    # Plot the forecast
    # im = ax_results[1].imshow(np.round(da_forecast.values), vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    im = ax_results[1].imshow(da_forecast.values, vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    plt.colorbar(im, ax = ax_results[1])
    ax_results[1].set_title('Forecast')
    # Plot the real image
    # im = ax_results[2].imshow(np.round(da_observed.values), vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    im = ax_results[2].imshow(da_observed.values, vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    plt.colorbar(im, ax=ax_results[2])
    ax_results[2].set_title('Real image')
    # # put critical stats info in a 4th subplot for easy access when evaluating data
    # ax_results[3].text(0.0, 0.0, info)
    fig_results.text(1.0, 0.5, info, ha='left', va='center')

    if myDate:
        if 'n' in myType:
            fig_results.suptitle(f'Neural Network: {myDate}')
        elif 'p' in myType:
            fig_results.suptitle(f'Polynomial: {myDate}')
        else:
            fig_results.suptitle(f'{myDate}')

    plt.tight_layout(rect=[0, 0, 0.95, 1])
    
    # display image
    plt.show()
    
    return overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, fig_results



# convert water mask dates to datetime64 dates
def get_WMDate(WMflnm, format='none'):
    # parse water mask name
    pathNflnm, file_extension = os.path.splitext(WMflnm)
    pieces_path = pathNflnm.split('/') # the pieces of the path. 
    pieces_flnm = pieces_path[-1].split('_')
    daDate = pieces_flnm[0]
    if format == 'none':
        WMDate = np.datetime64(daDate) # save as a datetime64
    elif format == 'dashed':
        WMDate = np.datetime64(daDate[0:4]+'-'+daDate[4:6]+'-'+daDate[6:]) # save as a datetime64

    return WMDate


# function to save statistics in a CSV file.
def stats2csv(flnm,header,stats,forecasting_file):
    with open(flnm,'w') as myFile:
        wr = csv.writer(myFile, quoting=csv.QUOTE_ALL)
        wr.writerows([header])
        wr.writerows(stats)
        wr.writerows([['File used for forecasting',forecasting_file]])




***
## Select date range of interest

### Selected hindcast

The user selects a particular day in the past to 'forecast'. 

In [ ]:
print(f'Date of Flood: {da_tot[ind_flood].time.values.astype('datetime64[D]')}')

In [ ]:
# input year, month, and day
myDate = '2023-06-11' # for bang-West & sylhet_central
# myDate = '2023-06-23' # for sylhet_central

numDays = 50 # number of days to forecast. 

myDate = np.datetime64(myDate)

In [ ]:
# Find the index of the first day of the forecast
ind_start = np.where(q_tot.time == myDate)[0][0]

# Find the index of the last day of the forecast
ind_stop = np.where(q_tot.time == myDate)[0][0]+numDays

# Crop the discharge to the same range as the forecast
q_forecast = q_tot.isel(time=slice(ind_start, ind_stop + 1))

In [ ]:
q_forecast.time

### Generalized forecast

This will acquire the 15 day forecast of geoglows data and forecast from today to 15 days from now. 

In [ ]:
ds = geoglows.data.forecast(ReachID, format='xarray')
ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))
ds = ds.resample(time='1D').sum()

In [ ]:
q_forecast = ds['flow_median'].rename('discharge')

### Generalized hindcast

In this, a 'forecast' is made for the 12 days including and prior to the desired flooding event to predict. This is generally used to assess the accuracy and precision of a given forecast model. 

In [ ]:
### --- Prepare Discharge dataset --- ###

# Find the index of the first day of the forecast
ind_start = np.where(q_tot.time == q.time[-1])[0][0]

# Find the index of the last day of the forecast
# ind_stop = np.where(q_tot.time == da_tot.time[ind_flood])[0][0]
ind_stop = np.where(q_tot.time == da_tot.time)[0][0]
ind_stop = ind_start + 15

# Crop the discharge to the same range as the forecast
q_forecast = q_tot.isel(time=slice(ind_start, ind_stop + 1)).resample(time='1D').mean()#.rolling(time=5).mean()
q_forecast

***
# Run the single highest-mode forecast

In [ ]:
### --- Run forecast on a single mode --- ###
# neural network fitting
myMode_nn = 1
if modes_nn != []:
    da_nn = myForecaster('neural network',modes_nn, myMode_nn, q_forecast, reof_ds, models_nn, type_file, da_tot, 'single')

In [ ]:
# polynomial fitting
myMode_poly = 1
if modes_poly != []:
    da_poly = myForecaster('polynomial', modes_poly, myMode_poly, q_forecast, reof_ds, polys, type_file, da_tot, 'single')

### Check the values of the forecast

The only unique values should be 0 (not water), 1 (water), and NaN (regions outside the region of interest).

In [ ]:
unique_elements_poly, counts_poly = np.unique(da_poly.values, return_counts=True)
unique_elements_nn,   counts_nn    = np.unique(da_nn.values  , return_counts=True)
print(f"Polynomial    : {unique_elements_poly}")
print(f"Neural Network: {unique_elements_nn  }")
print(f"counts_poly : {counts_poly}")
print(f"counts_nn   : {counts_nn  }")

***
# Multi-mode Forecast

This code is in progress. It runs, but the resultant forecasts are not as good as the single mode forecast. 

In [ ]:
### --- Set up variables & objects --- ###

# Get dimensions of datasets
n_modes, y, x = reof_ds.spatial_modes.shape
t = q_forecast.shape[0]

# Create the variable filled with zeros
zeros_variable = np.zeros((t, y, x))

### --- Initialize some important arrays --- ###

# Create the new DataArray with zeros_variable
da_nn_aggregate = xr.DataArray(
    zeros_variable,
    coords={'time': q_forecast.time, 'y': da_tot.y, 'x': da_tot.x},
    dims=['time', 'y', 'x']
)
da_poly_aggregate = xr.DataArray(
    zeros_variable,
    coords={'time': q_forecast.time, 'y': da_tot.y, 'x': da_tot.x},
    dims=['time', 'y', 'x']
)

type_file_sum = 'none' # making it anything other than 'Water_Masks' prevents the rounding from running, which we need to do after we've summed everything up. 



In [ ]:
### --- User selects modes to combine --- ###

print(f'modes_nn:   {modes_nn}\nmodes_poly: {modes_poly}')

In [ ]:
# Select which modes to include for the polynomial forecast
myModes_poly = modes_poly

In [ ]:
# Select which modes to include for the neural network forecast
myModes_nn = modes_nn

In [ ]:
### --- Run forecast on all of the best modes and sum them --- ###

# Run forecast for the polynomials
if myModes_poly != []:
    for mode in myModes_poly:
        da_poly_loop = myForecaster('polynomial', modes_poly, mode, q_forecast, reof_ds, polys, type_file_sum, da_tot, 'multi')
        da_poly_aggregate.values = da_poly_aggregate.values + da_poly_loop.values
    da_poly_aggregate.values = da_poly_aggregate.values + reof_ds.center.values 
    # change anything greater than or equal to 0.5 and less than 255 to 1 (water)
    idx = (da_poly_aggregate >= 0.5) * (da_poly_aggregate < 255)
    da_poly_aggregate.values[idx] = 1.0    
    # change anything less than 0.5 to zero (not water)
    idx = (da_poly_aggregate < 0.5)
    da_poly_aggregate.values[idx] = 0.0
    # change anything greater than or equal to 255 to nan
    idx = (da_poly_aggregate >= 255)
    da_poly_aggregate.values[idx] = np.nan
    # da_poly_aggregate.values = np.where(da_poly_aggregate.values==255, np.nan, da_poly_aggregate.values)

In [ ]:
# Run forecast for the neural network
if myModes_nn != []:
    for mode in myModes_nn:
        da_nn_loop = myForecaster('neural network',modes_nn, mode, q_forecast, reof_ds, models_nn, type_file_sum, da_tot, 'multi') # calculate the forecast for an individual mode
        # if mode == myModes_nn[0]: # for testing values returned
        #     da_nn_loops = da_nn_loop.copy(deep=True)
        # else:
        #     da_nn_loops = xr.concat([da_nn_loops, da_nn_loop], dim="mode")
        da_nn_aggregate.values = da_nn_aggregate.values + da_nn_loop.values # collect all of the forecasts
    da_nn_aggregate.values = da_nn_aggregate.values + reof_ds.center.values 
    # change anything greater than or equal to 0.5 and less than 255 to 1 (water)
    idx = (da_nn_aggregate >= 0.5) * (da_nn_aggregate < 255)
    da_nn_aggregate.values[idx] = 1.0
    # change anything less than 0.5 to zero (not water)
    idx = (da_nn_aggregate < 0.5)
    da_nn_aggregate.values[idx] = 0.0
    # change anything greater than or equal to 255 to nan
    idx = (da_nn_aggregate >= 255)
    da_nn_aggregate.values[idx] = np.nan
    # da_nn_aggregate.values = np.where(da_nn_aggregate.values==255, np.nan, da_nn_aggregate.values)

In [ ]:
unique_elements_poly_m, counts_poly_m = np.unique(da_poly_aggregate.values, return_counts=True)
unique_elements_nn_m,   counts_nn_m    = np.unique(da_nn_aggregate.values  , return_counts=True)
print(f"Polynomial_m    : {unique_elements_poly_m}")
print(f"Neural Network_m: {unique_elements_nn_m  }")
print(f"counts_poly_m : {counts_poly_m}")
print(f"counts_nn_m   : {counts_nn_m  }")

***
## Clip forecasts to relevant Region of Interest (ROI) (i.e., remove the buffer zone) (OPTIONAL)

The REOF suffers from edge effects. Edge effects reduce the accuracy and precision of forecasts. Because of this, it is a part of best practices to run the training on a region with a buffer zone which is then later removed. The required buffer zone is usually a few kilometers. 

This section of code clips the forecasts to the ROI. The ROI is usually defined as a watershed. 

In [ ]:
from osgeo import gdal
gdal.UseExceptions()
import glob

In [ ]:
# Write function to save the forecasts as geotiffs. 
def saveForecasts_v2(saveName, tiff, CRS, transform):
    # make directory if necessary
    directory = os.path.dirname(saveName)
    if not os.path.exists(directory):
        os.makedirs(directory)
    else:
        # print(f"Directory already exists: {directory}")
        pass
    
    myValues = np.abs(tiff.values) # need to take the absolute value as there are some "negative" zero values (I know, negative zero isn't actually a thing; the computer sure thinks it is)
    # open raster file and save
    with rasterio.open(
        saveName,
        mode="w",
        driver="GTiff",
        height=tiff.shape[0],
        width=tiff.shape[1],
        count=1,
        dtype=tiff.dtype,
        crs=CRS,
        transform=transform,
    ) as new_dataset:
        new_dataset.write(myValues, 1)

    return
    

In [ ]:
# Create debuffered forecasts and water masks. 

if debuff:     
    # Initialize important bits
    # get the shapefile info. 
    debuff_shapefile.selected
    shp = Path(debuff_shapefile.selected)
    if shp.suffix == '.shp':
        gInfo = gdal.OpenEx(str(shp))
        layer = gInfo.GetLayer()
        feature = layer.GetFeature(0)
        wkt = feature.GetGeometryRef().ExportToWkt()
    # set up folder names in which to save the files
    temp_folder = Path(fc.selected, 'debuffing')
    folder_debuffed = Path(fc.selected, 'debuffed')

    # create full list of debuffered files
    debuffeds_nn_sM   = []
    debuffeds_nn_mM   = []
    debuffeds_poly_sM = []
    debuffeds_poly_mM = []
    
    ### --- Subset datacheck, forecasts, and water masks --- ### 
    # First, create new datacheck and forecasts that are subsetted to the relevant region of interest. 
    os.makedirs(temp_folder, exist_ok=True)
    os.makedirs(folder_debuffed, exist_ok=True)
    # save the data check (da_tot)
    saveName_dataCheck = Path(temp_folder,'dataCheck.tif')
    saveForecasts_v2( saveName_dataCheck, da_tot, CRS, transform)
    debuffed_dataCheck = Path(folder_debuffed,'dataCheck.tif')
    gdal.Warp(str(debuffed_dataCheck), str(saveName_dataCheck), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
    # Subset the relevant files (the forecasts, and water masks)
    # subset the forecasts
    county = 0
    for tiff_nn_sM, tiff_nn_mM, tiff_poly_sM, tiff_poly_mM in zip(da_nn, da_nn_aggregate, da_poly, da_poly_aggregate):
        # create save name for 'debuffering' files
        fileName=f'Day-{county:03d}_{str(tiff_nn_sM.time.dt.strftime("%Y%m%d").values)}.tif'
        # print(fileName)
        # Save the forecasts (da_nn, da_nn_aggregate, da_poly, da_poly_aggregate)
        # current full save name for 'debuffering' files
        saveName_nn_sM   = Path(temp_folder, 'nn_sM',   fileName)
        saveName_nn_mM   = Path(temp_folder, 'nn_mM',   fileName)
        saveName_poly_sM = Path(temp_folder, 'poly_sM', fileName)
        saveName_poly_mM = Path(temp_folder, 'poly_mM', fileName)
        saveForecasts_v2(saveName_nn_sM,   tiff_nn_sM,   CRS, transform)
        saveForecasts_v2(saveName_nn_mM,   tiff_nn_mM,   CRS, transform)
        saveForecasts_v2(saveName_poly_sM, tiff_poly_sM, CRS, transform)
        saveForecasts_v2(saveName_poly_mM, tiff_poly_mM, CRS, transform)
        # Subset 'debuffering' files and save them in 'debuffed'
        # set up file names
        debuffed_nn_sM   = Path(folder_debuffed, 'nn_sM',   fileName)
        debuffed_nn_mM   = Path(folder_debuffed, 'nn_mM',   fileName)
        debuffed_poly_sM = Path(folder_debuffed, 'poly_sM', fileName)
        debuffed_poly_mM = Path(folder_debuffed, 'poly_mM', fileName)
        
        debuffeds_nn_sM.append(  debuffed_nn_sM)
        debuffeds_nn_mM.append(  debuffed_nn_mM)
        debuffeds_poly_sM.append(debuffed_poly_sM)
        debuffeds_poly_mM.append(debuffed_poly_mM)
        # make directory if it doesn't exist
        if not os.path.exists( os.path.dirname(debuffed_nn_sM) ):
            os.makedirs( os.path.dirname(debuffed_nn_sM) )
        if not os.path.exists( os.path.dirname(debuffed_nn_mM) ):
            os.makedirs( os.path.dirname(debuffed_nn_mM) )
        if not os.path.exists( os.path.dirname(debuffed_poly_sM) ):
            os.makedirs( os.path.dirname(debuffed_poly_sM) )
        if not os.path.exists( os.path.dirname(debuffed_poly_mM) ):
            os.makedirs( os.path.dirname(debuffed_poly_mM) )
        # use gdal to create subsets
        gdal.Warp(str(debuffed_nn_sM),   str(saveName_nn_sM),   dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_nn_mM),   str(saveName_nn_mM),   dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_poly_sM), str(saveName_poly_sM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_poly_mM), str(saveName_poly_mM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        county = county + 1
    # subset the water masks
    WMs = glob.glob(fc.selected + 'Water_Masks/*combined.tif*')
    WMs.sort()
    WMDates = []
    for WM in WMs:
        WMDate = get_WMDate(WM)
        WMDates.append(WMDate)
        debuffed_WM = Path(folder_debuffed, 'Water_Masks', str(WMDate).replace("-","")+'_water_mask_combined.tif')
        if not os.path.exists( os.path.dirname(debuffed_WM) ):
            os.makedirs( os.path.dirname(debuffed_WM) )
        gdal.Warp(str(debuffed_WM), str(WM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)

    # delete the temporary folder to save space. 
    import shutil
    shutil.rmtree(temp_folder)

In [ ]:
### --- Get Dates from filenames --- ###
def get_dates(dir_path):
    dates = []
    pths = list(dir_path.glob(f'*.tif*'))

    for p in pths:
        date_regex = r'\d{8}'
        date = re.search(date_regex, str(p))
        if date:
            dates.append(date.group(0))
    return dates





### --- Load Geotiffs Function --- ###
def load_tiffs(folder, stop_ind = -1):

    # Gather names of files corresponding to the file type and polarization we want
    tiff_dir = Path(folder)
    tiffs = list(tiff_dir.glob(f'*.tif*'))
    
    # Gather the date of each file
    times = get_dates(tiff_dir)
    
    # Create a list of indices based on the sorted order of times
    sorted_indices = sorted(range(len(times)), key=lambda i: times[i])
    
    # Sort the paths based on the times
    tiffs = [tiffs[i] for i in sorted_indices]
    
    # Sort the times axisdd
    times.sort()
    times = pd.DatetimeIndex(times)
    times.name = "time"
    
    # Create the dataset gathering the input images
    da = xr.concat([rxr.open_rasterio(f).squeeze(dim='band') for f in tiffs[:stop_ind]], dim=times[:stop_ind])
    da = da.drop_vars(['band', 'spatial_ref'])

    print(f"Dataset size in memory: {da.nbytes / 1e6:.2f} MB")

    return da, tiffs, times

In [ ]:
# load debuffered forecasts and water masks

if debuff:
    # load subsetted dataCheck in relevant object
    # da_tot_check = xr.concat([rxr.open_rasterio(dataCheck).squeeze(dim='band') for dataCheck in debuffed_dataCheck, dim=da_tot.time)
    da_tot_check = rxr.open_rasterio(debuffed_dataCheck) # THIS partially WORKS; loads data, but doesn't load time
    da_tot_check = da_tot_check.drop_vars(['band', 'spatial_ref'])
    da_tot_check = da_tot_check[0]
    da_tot_check['time'] = da_tot.time
    # delete original and replace it
    del da_tot
    da_tot = da_tot_check.copy(deep=True)
    
    # Load subsetted forecast values into temporary objects
    # create directory names
    nn_sM_dir   = Path(folder_debuffed, 'nn_sM')
    nn_mM_dir   = Path(folder_debuffed, 'nn_mM')
    poly_sM_dir = Path(folder_debuffed, 'poly_sM')
    poly_mM_dir = Path(folder_debuffed, 'poly_mM')
    # load subsetted forecasts 
    da_nn_sM_check  , _, _ = load_tiffs( nn_sM_dir )
    print('Debuffered single mode neural network forecasts loaded.')
    da_nn_mM_check  , _, _ = load_tiffs( nn_mM_dir )
    print('Debuffered multi mode neural network forecasts loaded.')
    da_poly_sM_check, _, _ = load_tiffs( poly_sM_dir )
    print('Debuffered single mode polynomial forecasts loaded.')
    da_poly_mM_check, _, _ = load_tiffs( poly_mM_dir )
    print('Debuffered multi mode polynomial forecasts loaded.')
    
    # Reassign subsetted forecasts (new, temporary objects) to appropriate variables
    # delete original objects and replace them (this is so they keep the same name and I don't have to change a bunch of other code)
    del da_nn, da_nn_aggregate, da_poly, da_poly_aggregate
    da_nn             = da_nn_sM_check.copy(deep=True)
    da_nn_aggregate   = da_nn_mM_check.copy(deep=True)
    da_poly           = da_poly_sM_check.copy(deep=True)
    da_poly_aggregate = da_poly_mM_check.copy(deep=True)
    
    # delete temporary variables
    del da_tot_check, da_nn_sM_check, da_nn_mM_check, da_poly_sM_check, da_poly_mM_check

***
## Accuracy of forecasts

Compare the forecast results to actual flood extent or SAR imagery.

Note: This is for ONLY the flood chosen in in the previous notebook. The next visualization will compare produced forecasts to all extant water maps. 


In [ ]:
myColorScheme = 'seismic_r' #'gray' # 'seismic'
print('Single Mode - Neural Network')
overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFigs_singleMode_nn = forecast2observed_comparison(da_nn[12], da_tot, myColorScheme,'nn', da_nn[12].time.dt.strftime("%Y-%m-%d").values)
print('Multi Mode - Neural Network')
overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFigs_multiMode_nn = forecast2observed_comparison(da_nn_aggregate[12], da_tot, myColorScheme,'nn', da_nn_aggregate[12].time.dt.strftime("%Y-%m-%d").values)
print('Single Mode - Polynomial')
overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFigs_singleMode_poly = forecast2observed_comparison(da_poly[12], da_tot, myColorScheme,'poly', da_poly[12].time.dt.strftime("%Y-%m-%d").values)
print('Multi Mode - Polynomial')
overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFigs_multiMode_poly = forecast2observed_comparison(da_poly_aggregate[12], da_tot, myColorScheme,'poly', da_poly_aggregate[12].time.dt.strftime("%Y-%m-%d").values)

## Compare long-term forecast

This section of code compares the long-term forecast (several months to years worth of forecasts) to historical SAR-derived water extents. 

In [ ]:
import glob
WMs = glob.glob(fc.selected + 'Water_Masks/*combined.tif*')
WMs.sort()

In [ ]:
### --- Single Mode --- ###

statistics_singleMode_poly = []
statistics_singleMode_nn   = []
statistics_header = ['Date', 'Overall Accuracy', 'Precision', 'CSI', 'Recall', 'Difference Summation', 'Difference Percentile']
myFigs_singleMode_nn = []
myFigs_singleMode_poly = []
for WM in WMs:    
    WMDate = get_WMDate(WM, format='dashed')
    try: 
        # Get index of forecasts that matches the water mask
        idx = np.where(da_poly.time.values == WMDate)[0][0] # this will return an error if a match isn't found, so I have it in a 'try' statement. FYI, this makes finding errors impossible! 
        print(f'date2: {WMDate}')
        # if idx == 0:
        #     print('break!')
        #     break
    
        # Load water mask
        water_mask = rxr.open_rasterio(WM)
        water_mask = water_mask.drop_vars(['band', 'spatial_ref'])                               
        water_mask = water_mask[0]
    
        # Run comparison between forecast and water mask
        overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFig_singleMode_nn = forecast2observed_comparison(da_nn[idx], water_mask, myColorScheme, 'nn', WMDate)
        statistics_singleMode_nn.append([str(WMDate), overall_accuracy, precision, CSI, recall, difference_sum, difference_percent])
        myFigs_singleMode_nn.append(myFig_singleMode_nn)
        
        overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFig_singleMode_poly = forecast2observed_comparison(da_poly[idx], water_mask, myColorScheme, 'poly', WMDate)
        statistics_singleMode_poly.append([str(WMDate), overall_accuracy, precision, CSI, recall, difference_sum, difference_percent])
        myFigs_singleMode_poly.append(myFig_singleMode_poly)
    except:
        pass


In [ ]:
### --- Multi Mode --- ###
myColorScheme = 'seismic_r'
statistics_multiMode_poly = []
statistics_multiMode_nn   = []
statistics_header = ['Date', 'Overall Accuracy', 'Precision', 'CSI', 'Recall', 'Difference Summation', 'Difference Percentile']
myFigs_multiMode_nn = []
myFigs_multiMode_poly = []
for WM in WMs:
    WMDate = get_WMDate(WM, format='dashed')
    try:
        # Get index of forecasts that matches the water mask
        idx = np.where(da_poly.time.values == WMDate)[0][0] # this will return an error if a match isn't found, so I have it in a 'try' statement. FYI, this makes finding errors impossible! 

        # Load water mask
        water_mask = rxr.open_rasterio(WM)
        water_mask = water_mask.drop_vars(['band', 'spatial_ref'])                               
        water_mask = water_mask[0]
    
        # Run comparison between forecast and water mask
        overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFig_multiMode_nn = forecast2observed_comparison(da_nn_aggregate[idx], water_mask, myColorScheme, 'nn', WMDate)
        statistics_multiMode_nn.append([str(WMDate), overall_accuracy, precision, CSI, recall, difference_sum, difference_percent])
        myFigs_multiMode_nn.append(myFig_multiMode_nn)
        
        overall_accuracy, CSI, precision, recall, difference_sum, difference_percent, myFig_multiMode_poly = forecast2observed_comparison(da_poly_aggregate[idx], water_mask, myColorScheme, 'poly', WMDate)
        statistics_multiMode_poly.append([str(WMDate), overall_accuracy, precision, CSI, recall, difference_sum, difference_percent])
        myFigs_multiMode_poly.append(myFig_multiMode_poly)
    except:
        pass


***
### Plot of flood forecasts, flood percentages, and discharge through time

In [ ]:
# get the flood percentages
flood_percent_nn = floodPercentCalc(da_nn)
flood_percent_nn_aggregate = floodPercentCalc(da_nn_aggregate)
flood_percent_poly = floodPercentCalc(da_poly)
flood_percent_poly_aggregate = floodPercentCalc(da_poly_aggregate)

In [ ]:
fp_axis_lims

In [ ]:
# Neural Network Fit
myFldPcnt_singleMode_nn = plot_flood_and_discharge(da_nn, flood_percent_nn, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)
myFldPcnt_multiMode_nn = plot_flood_and_discharge(da_nn_aggregate, flood_percent_nn_aggregate, q_forecast, floodpercent, time_flood, fp_lims = fp_axis_lims)

In [ ]:
# Polynomial Fit
myFldPcnt_singleMode_poly = plot_flood_and_discharge(da_poly, flood_percent_poly, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)
myFldPcnt_multiMode_poly = plot_flood_and_discharge(da_poly_aggregate, flood_percent_poly_aggregate, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)

***
## Plot Forecasted versus Observed Flood Percentages

The below code will produce a graph of Forecasted (x-axis) vs Observed (y-axis) flood percentages for each of the single/multi and neural network/polynomial fits. 

A black line will mark the ideal match (i.e., a perfect one). Markers to the right mean the forecast overpredicts water extent; to the left means an underprediction. 

In [ ]:
def realVSobservedFloodPercentages(time_forecasts, floodPercents_forecast, time_real, floodPercents_real, title = None):

    forRMSE = []
    
    plt.figure()

    for i in range(0,len(floodPercents_forecast)):
        try: 
            idx = np.where(time_forecasts[i] == time_real)[0][0]
            plt.scatter(floodPercents_forecast[i], floodPercents_real[idx], color='red', s=50)
            forRMSE.append(floodPercents_forecast[i] - floodPercents_real[idx])
        except: 
            continue

    # add labels (and a title if there is text for it)
    plt.xlabel('Forecasted Flood Percentage')
    plt.ylabel('Observed Flood Percentage')
    if title:
        plt.title(title)

    # plot straight line to make clear what is an over/under forecast. 
    max_fP = max(floodPercents_real)
    if max(floodPercents_forecast) > max_fP:
        max_fP = max(floodPercents_forecast)
    plt.plot([0, max_fP], [0, max_fP], color='black')
    plt.grid()
    ax = plt.gca()
    ax.set_aspect('equal')
    # print(f'county = {county}')

    # calculate and display RMSE
    myRMSE = np.sqrt( np.mean( np.square(forRMSE) ) )
    plt.text(max_fP/4, max_fP*(3/4), f'RMSE = {myRMSE:.1f}')

In [ ]:
realVSobservedFloodPercentages(da_poly_aggregate.time.values, flood_percent_poly_aggregate, time_flood, floodpercent, 'Multi Mode Polynomial')
realVSobservedFloodPercentages(da_poly.time.values, flood_percent_poly, time_flood, floodpercent, 'Single Mode Polynomial')
realVSobservedFloodPercentages(da_nn_aggregate.time.values, flood_percent_nn_aggregate, time_flood, floodpercent, 'Multi Mode Neural Network')
realVSobservedFloodPercentages(da_nn.time.values, flood_percent_nn, time_flood, floodpercent, 'Single Mode Neural Network')

In [ ]:
### --- Save visualizations and statistics --- ###


# set up save location names
saveLoc_sM_nn   = Path(fc.selected, 'Figures',myLoc,f'singleMode_nn_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_mM_nn   = Path(fc.selected, 'Figures',myLoc,f'multiMode_nn_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_sM_poly = Path(fc.selected, 'Figures',myLoc,f'singleMode_poly_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_mM_poly = Path(fc.selected, 'Figures',myLoc,f'multiMode_poly_smoothing-{smoothing_frame[myIdx]:02d}days')

# creat save locations if they don't exist
saveLoc_sM_nn.mkdir(parents=True, exist_ok=True)
saveLoc_mM_nn.mkdir(parents=True, exist_ok=True)
saveLoc_sM_poly.mkdir(parents=True, exist_ok=True)
saveLoc_mM_poly.mkdir(parents=True, exist_ok=True)

# save the figures for later display
for i in range(0,len(myFigs_singleMode_nn)): 
    # set up file name for each type
    flnm_sM_nn   = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_singleMode_nn[i][0]}.png'
    flnm_mM_nn   = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_nn[i][0]}.png'
    flnm_sM_poly = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_poly[i][0]}.png'
    flnm_mM_poly = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_poly[i][0]}.png'
    # save figures
    myFigs_singleMode_nn[i].savefig(Path(saveLoc_sM_nn,flnm_sM_nn), bbox_inches='tight')
    myFigs_multiMode_nn[i].savefig(Path(saveLoc_mM_nn,flnm_mM_nn), bbox_inches='tight')
    myFigs_singleMode_poly[i].savefig(Path(saveLoc_sM_poly,flnm_sM_poly), bbox_inches='tight')
    myFigs_multiMode_poly[i].savefig(Path(saveLoc_mM_poly,flnm_sM_poly), bbox_inches='tight')

# save the statistics for later inspection
stats2csv( Path(saveLoc_sM_nn,'statistics_singleMode_nn.csv'), statistics_header, statistics_singleMode_nn, forecasting_files)
stats2csv( Path(saveLoc_mM_nn,'statistics_multiMode_nn.csv'), statistics_header, statistics_multiMode_nn, forecasting_files)
stats2csv( Path(saveLoc_sM_poly,'statistics_singleMode_poly.csv'), statistics_header, statistics_singleMode_poly, forecasting_files)
stats2csv( Path(saveLoc_mM_poly,'statistics_multiMode_poly.csv'), statistics_header, statistics_multiMode_poly, forecasting_files)

# save flood percentage figures
myFldPcnt_singleMode_nn.savefig(Path(saveLoc_sM_nn,'floodPercent_sM_nn.png'))
myFldPcnt_multiMode_nn.savefig(Path(saveLoc_mM_nn,'floodPercent_mM_nn.png'))
myFldPcnt_singleMode_poly.savefig(Path(saveLoc_sM_poly,'floodPercent_sM_poly.png'))
myFldPcnt_multiMode_poly.savefig(Path(saveLoc_mM_poly,'floodPercent_mM_poly.png'))

***
## Create, save, and display GIF of forecast results

In [ ]:
# Function to create the animated gifs
import matplotlib.animation as animation
def animate(dataset, fps, flnm, type_file, rewrite=False):
    
    if os.path.exists(flnm) and rewrite==False:
        print("GIF already exists.")
    else:
        # Create the animation
        fig = plt.figure()
        num_frames = dataset.shape[0]-1 # Because of Python's indexing and HTML indexing difference
        ani = animation.FuncAnimation(fig, update_plot, frames=num_frames, fargs=(dataset,type_file,), blit=False)

        # To show the animation in Jupyter Notebook (optional)

        from IPython.display import HTML
        HTML(ani.to_jshtml())

        # Save the animation as a GIF
        ani.save(flnm, writer='pillow', fps=fps)
        plt.close(ani._fig)
        plt.close(fig)

# Function that plots one image at a time
def update_plot(frame, dataset, type_file):
    # Extract the 2D slice corresponding to the current time step
    current_slice = dataset[frame, :, :]
    
    # Clear the previous plot
    plt.clf()
    
    # Depending on if your input iw water mask or RTC, the colormap changes
    if type_file == 'Water_Masks':
        plt.imshow(current_slice, cmap='Blues', interpolation="none", vmin=0, vmax=1.2)
    else:
        # Plot the 2D slice
        plt.imshow(current_slice, cmap='viridis', vmin = dataset.mean() - dataset.std(), vmax = dataset.mean() + dataset.std())
    
    # Add a title and other plot decorations (optional)
    plt.colorbar()
    
    # Add the date as a subtitle
    current_date = str(dataset.time[frame].values)[:10]  # Assuming time is in datetime format
    plt.text(0.5, 1.01, f'Date: {current_date}', transform=plt.gca().transAxes,
             ha='center', va='bottom', fontsize=10)

In [ ]:
# Production! 
# set up file save names
flnm_sM_poly_gif = Path(saveLoc_sM_poly,'sM-poly_gif-forecast_.gif')
flnm_mM_poly_gif = Path(saveLoc_mM_poly,'mM-poly_gif-forecast_.gif')
flnm_sM_nn_gif = Path(saveLoc_sM_nn,'sM-nn_gif-forecast_.gif')
flnm_mM_nn_gif = Path(saveLoc_mM_nn,'mM-nn_gif-forecast_.gif')

# set number of frames per second
fps = 2 # frames per second

# save the gifs
animate(da_poly[0:], fps, flnm_sM_poly_gif, 'Water_Masks', rewrite=True)
print('Single mode poly complete.')
animate(da_poly_aggregate[0:], fps, flnm_mM_poly_gif, 'Water_Masks', rewrite=True)
print('Multi mode poly complete.')
animate(da_nn[0:], fps, flnm_sM_nn_gif, 'Water_Masks', rewrite=True)
print('Single mode NN complete.')
animate(da_nn_aggregate[0:], fps, flnm_mM_nn_gif, 'Water_Masks', rewrite=True)
print('Multi mode NN complete.')